In [43]:
import re, time, warnings, copy, gc
from pathlib import Path
from functools import partial
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nibabel as nib
import nibabel.orientations as nibo
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math
from scipy.ndimage import rotate as scipy_rotate
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
import subprocess, shutil, tempfile
from scipy import ndimage

torch.backends.cudnn.benchmark = True
from monai.config import print_config
from monai.utils import set_determinism
from monai.data import (
    Dataset, DataLoader, decollate_batch,
    pad_list_data_collate, set_track_meta, MetaTensor,
)
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, CropForegroundd, AsDiscrete, Invertd,
)
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.utils.enums import MetricReduction
from monai.inferers import sliding_window_inference
from monai.networks.nets import SwinUNETR as _SwinUNETR

# CRITICAL: enables transform metadata tracking so Invertd can undo every step
set_track_meta(True)

DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()

warnings.filterwarnings('ignore')


In [44]:
# ── Configuration ────────────────────────────────────────────────────
DATA_ROOT = Path(r"C:\Users\DELL\Desktop\BraTS26\All_data_raw\BraTS26_PED_validation\BraTS26_PED_validation")
OUTPUT_ROOT   = Path(r"C:\Users\DELL\Desktop\BraTS26\test_docker")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ── Checkpoint paths (one per specialist) ─────────────────────────────────────
CKPT_ET  = Path(r"C:\Users\DELL\Desktop\BraTS26\BraTS'26-Task_02\Final_Models\best_swinunetr_et_new.pth")
CKPT_NET = Path(r"C:\Users\DELL\Desktop\BraTS26\BraTS'26-Task_02\Final_Models\best_swinunetr_net_new.pth")
CKPT_ED  = Path(r"C:\Users\DELL\Desktop\BraTS26\BraTS'26-Task_02\Final_Models\best_3dunet_ed.pth")
CKPT_CC  = Path(r"C:\Users\DELL\Desktop\BraTS26\BraTS'26-Task_02\Final_Models\best_3dunet_cc.pth")

SEED = 42
set_determinism(seed=SEED)
USE_AMP    = True
IMAGE_KEYS = ['t1n', 't1c', 't2w', 't2f']
LABEL_NAMES = {0: 'BG', 1: 'ET', 2: 'NET', 3: 'CC', 4: 'ED'}
NUM_CLASSES = 5

# ── Architecture constants (verified from each training notebook) ──────────────
# Shared across all 4 models
SWIN_FEATURE_SIZE = 48
SWIN_DEPTHS       = (2, 2, 2, 2)
SWIN_NUM_HEADS    = (3, 6, 12, 24)
SWIN_DROP_RATE    = 0.0
SWIN_ATTN_DROP    = 0.0
SWIN_DROP_PATH    = 0.0

# Per-model PATCH_SIZE (= img_size used during training) and SW_ROI
ED_PATCH   = (96,  96,  96);   ED_SW_ROI  = (96,  96,  96);   ED_SW_OVERLAP  = 0.25; ED_SW_BATCH  = 4
CC_PATCH   =(64,  64,  64);   CC_SW_ROI  = (64,  64,  64);   CC_SW_OVERLAP  = 0.50; CC_SW_BATCH  = 4
ET_PATCH   = (96,  96,  96);   ET_SW_ROI  = (96, 96, 96);  ET_SW_OVERLAP  = 0.50; ET_SW_BATCH  = 4
NET_PATCH  = (96,  96,  96);   NET_SW_ROI = (96,  96,  96);   NET_SW_OVERLAP = 0.25; NET_SW_BATCH = 4

# ── Output sub-folders ─────────────────────────────────────────────────────────
DIR_FULL  = OUTPUT_ROOT / 'pred_full'

for d in [DIR_FULL]:
    d.mkdir(parents=True, exist_ok=True)


# ── Architecture constants (verified from training notebooks) ─────────────────
UNET_CHANNELS = (32, 64, 128, 256, 320)
UNET_STRIDES  = (2, 2, 2, 2)
NUM_RES_UNITS = 2
UNET_IN_CH    = 4
UNET_OUT_CH   = 2   # binary: BG + one tumour class (all 4 specialist models)

# ── Sliding-window settings (matching training exactly) ──────────────────────
for name, p in [('DATA_ROOT', DATA_ROOT), ('OUTPUT_ROOT', OUTPUT_ROOT),
                ('CKPT_ED', CKPT_ED), ('CKPT_CC', CKPT_CC),
                ('CKPT_ET', CKPT_ET), ('CKPT_NET', CKPT_NET)]:
    print(f'')


In [45]:
# ── Model Definitions ───────────────────────────────────────────────

# ════════ MODEL 1 — CBAMUNet3D  (ED specialist) ══════════════════════════════
class ChannelAttention3D(nn.Module):
    def __init__(self, channels, ratio=16):
        super().__init__()
        mid = max(channels // ratio, 1)
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        scale = self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))
        return x * scale.view(scale.size(0), -1, 1, 1, 1)

class SpatialAttention3D(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv3d(2, 1, kernel_size=kernel_size,
                                  padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        cat = torch.cat([x.mean(dim=1, keepdim=True),
                          x.max(dim=1, keepdim=True)[0]], dim=1)
        return x * self.sigmoid(self.conv(cat))

class CBAM3D(nn.Module):
    def __init__(self, channels, ratio=16, spatial_kernel=7):
        super().__init__()
        self.ch = ChannelAttention3D(channels, ratio)
        self.sp = SpatialAttention3D(spatial_kernel)
    def forward(self, x):
        return self.sp(self.ch(x))

class CBAMUNet3D(nn.Module):
    """ED specialist. Trained with FocalTverskyLoss + DiceCELoss.
    Crop source during training: T2f (FLAIR)."""
    def __init__(self, in_channels=4, out_channels=2,
                  channels=(32,64,128,256,320), strides=(2,2,2,2), num_res_units=2):
        super().__init__()
        self.backbone = UNet(
            spatial_dims=3, in_channels=in_channels, out_channels=out_channels,
            channels=channels, strides=strides, num_res_units=num_res_units,
            norm=Norm.INSTANCE, dropout=0.0,
        )
    def forward(self, x):
        return self.backbone(x)

# ════════ MODEL 2 — SEUNet3D  (CC specialist) ════════════════════════════════
class SEBlock3D(nn.Module):
    def __init__(self, channels, ratio=16):
        super().__init__()
        mid = max(channels // ratio, 1)
        self.squeeze    = nn.AdaptiveAvgPool3d(1)
        self.excitation = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        scale = self.excitation(self.squeeze(x))
        return x * scale.view(scale.size(0), -1, 1, 1, 1)

class SEUNet3D(nn.Module):
    """CC specialist. Trained with FocalDice + WeightedBCE.
    Crop source during training: T2w."""
    def __init__(self, in_channels=4, out_channels=2,
                  channels=(32,64,128,256,320), strides=(2,2,2,2), num_res_units=2):
        super().__init__()
        self.backbone = UNet(
            spatial_dims=3, in_channels=in_channels, out_channels=out_channels,
            channels=channels, strides=strides, num_res_units=num_res_units,
            norm=Norm.INSTANCE, dropout=0.0,
        )
        self.se_blocks = nn.ModuleList([SEBlock3D(c) for c in channels[:-1]])
    def forward(self, x):
        return self.backbone(x)
        
# ── Cell 3 — SwinUNETRWrapper (identical to all 4 training notebooks) ─────────


class SwinUNETRWrapper(nn.Module):
    
    def __init__(
        self,
        in_channels: int = 4,
        out_channels: int = 2,
        img_size=None,
        feature_size: int = 48,
        depths=(2, 2, 2, 2),
        num_heads=(3, 6, 12, 24),
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        dropout_path_rate: float = 0.0,
        use_checkpoint: bool = False,   # False at inference
    ):
        super().__init__()
        self.net = _SwinUNETR(
            
            in_channels        = in_channels,
            out_channels       = out_channels,
            feature_size       = feature_size,
            depths             = depths,
            num_heads          = num_heads,
            drop_rate          = drop_rate,
            attn_drop_rate     = attn_drop_rate,
            dropout_path_rate  = dropout_path_rate,
            use_checkpoint     = use_checkpoint,
            spatial_dims       = 3,
        )

    def forward(self, x):
        return self.net(x)

def build_specialist(patch_size, out_channels=2):
    """Build one binary SwinUNETRWrapper matched to its training patch size."""
    return SwinUNETRWrapper(
        in_channels       = 4,
        out_channels      = out_channels,
        img_size          = patch_size,
        feature_size      = SWIN_FEATURE_SIZE,
        depths            = SWIN_DEPTHS,
        num_heads         = SWIN_NUM_HEADS,
        drop_rate         = SWIN_DROP_RATE,
        attn_drop_rate    = SWIN_ATTN_DROP,
        dropout_path_rate = SWIN_DROP_PATH,
        use_checkpoint    = False,
    )


    



In [46]:
# ── Load Weights ────────────────────────────────────────────────────
def _load_checkpoint(model, ckpt_path, label):
    ckpt_path = Path(ckpt_path)
    assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'
    raw = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    sd = raw.get('state_dict', raw) if isinstance(raw, dict) else raw

    clean = {}
    for k, v in sd.items():
        nk = k.replace('_orig_mod.', '').replace('module.', '')
        nk = re.sub(r'\.sub(\d+)', r'.submodule.\1', nk)
        nk = nk.replace('.subconv.', '.submodule.conv.')
        nk = nk.replace('.subresidual.', '.submodule.residual.')
        # CBAM key normalisation (ETUNet3D / NETUNet3D training used channel_att/spatial_att)
        nk = nk.replace('.channel_att.', '.ch.')
        nk = nk.replace('.spatial_att.', '.sp.')
        clean[nk] = v

    missing, unexpected = model.load_state_dict(clean, strict=True)
    if missing:    raise RuntimeError(f'{label}: missing keys: {missing[:5]}')
    if unexpected: raise RuntimeError(f'{label}: unexpected keys: {unexpected[:5]}')
    model.to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters())
    return model

model_ed = _load_checkpoint(
    CBAMUNet3D(in_channels=UNET_IN_CH, out_channels=UNET_OUT_CH,
               channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=NUM_RES_UNITS),
    CKPT_ED, 'CBAMUNet3D  [ED,  crop=t2f]'
)
model_cc = _load_checkpoint(
    SEUNet3D(in_channels=UNET_IN_CH, out_channels=UNET_OUT_CH,
             channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=NUM_RES_UNITS),
    CKPT_CC, 'SEUNet3D    [CC,  crop=t2w]'
)


model_net = _load_checkpoint(build_specialist(NET_PATCH),CKPT_NET,'SwinUNETR [NET, crop=t1c, patch=96³]')
model_et  = _load_checkpoint(build_specialist(ET_PATCH),  CKPT_ET,  'SwinUNETR  [ET,  crop=t1c, patch=96³]')

In [47]:
# ── Data Discovery ───────────────────────────────────────────────────
def find_modality(case_dir, key):
    hits = sorted(case_dir.glob(f'*-{key}.nii.gz'))
    if not hits:
        hits = sorted([
            p for p in case_dir.glob(f'*{key}*.nii*')
            if 'mask' not in p.name.lower()
        ])
    return str(hits[0]) if hits else None
labeled_dicts   = []
unlabeled_dicts = []
skipped = []

for d in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    sid  = d.name
    mods = {k: find_modality(d, k) for k in IMAGE_KEYS}
    miss = [k for k, v in mods.items() if v is None]
    if miss:
        skipped.append((sid, f'missing: {miss}')); continue
    entry = {**mods, 'subject_id': sid}
    seg   = sorted(d.glob('*-seg.nii.gz')) or sorted(d.glob('*seg*.nii*'))
    if seg:
        entry['seg'] = str(seg[0])
        labeled_dicts.append(entry)
    else:
        unlabeled_dicts.append(entry)

all_dicts = labeled_dicts + unlabeled_dicts

for sid, r in skipped:
    print(f'  Skipped {sid}: {r}')
if not all_dicts:
    raise RuntimeError(f'No cases found under {DATA_ROOT}')

ex = all_dicts[0]
for k in IMAGE_KEYS:
    orig = nib.load(ex[k])


In [48]:
#  In-Memory Skull Stripping ──────────────────────────────────────


# ── Config — change only these two if needed ──────────────────────────────────
USE_DOCKER   = True   # flip to False if FreeSurfer CLI is available
SKULL_BORDER = 1       # SynthStrip --border (mm)
SKULL_NO_CSF = False   # True = tighter mask (excludes CSF)

VAL_DATA_ROOT = DATA_ROOT

# ── Core skull-strip function ─────────────────────────────────────────────────

def deskull_case(entry: dict,
                 image_keys=IMAGE_KEYS,
                 mask_modality: str = 't1c',
                 border: int = SKULL_BORDER,
                 no_csf: bool = SKULL_NO_CSF,
                 use_docker: bool = USE_DOCKER) -> dict:
    
    sid     = entry['subject_id']
    tmp_dir = Path(tempfile.mkdtemp(prefix=f'skull_{sid}_'))
    new_entry = {k: v for k, v in entry.items()}   # copy seg, subject_id, etc.

    # ── Step 1: run SynthStrip on the mask modality ───────────────────────────
    mask_input = Path(entry[mask_modality])
    tmp_mask   = tmp_dir / 'brain_mask.nii.gz'

    if use_docker:
        # Mount only the case folder; SynthStrip reads/writes inside /data
        host_dir   = mask_input.parent
        cont_in    = f'/data/{mask_input.name}'
        cont_mask  = f'/data/{tmp_mask.name}'
        # We only need the mask (-m), not the stripped image (-o)
        # But SynthStrip requires -o even if we don't use it → write to tmp
        cont_brain = f'/data/_brain_throwaway.nii.gz'
        
        cmd = [
            'docker', 'run', '--rm',
            '-v', f'{host_dir}:/data',           # mount input folder
            '-v', f'{tmp_dir}:/tmp_out',          # mount temp output folder
            'freesurfer/synthstrip',
            '-i', cont_in,
            '-o', f'/tmp_out/_brain_throwaway.nii.gz',
            '-m', f'/tmp_out/brain_mask.nii.gz',
            '--border', str(border),
            
        ]
        if no_csf:
            cmd.append('--no-csf')
    else:
        tmp_brain_throwaway = tmp_dir / '_brain_throwaway.nii.gz'
        cmd = [
            'mri_synthstrip',
            '-i', str(mask_input),
            '-o', str(tmp_brain_throwaway),   # required even if unused
            '-m', str(tmp_mask),
            '--border', str(border),
        ]
        if no_csf:
            cmd.append('--no-csf')

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not tmp_mask.exists():
        shutil.rmtree(tmp_dir, ignore_errors=True)
        raise RuntimeError(
            f'[deskull_case] SynthStrip failed for {sid}\n'
            f'  CMD: {" ".join(cmd)}\n'
            f'  STDOUT: {result.stdout[-500:]}\n'
            f'  STDERR: {result.stderr[-500:]}'
        )

    # ── Step 2: load mask once ────────────────────────────────────────────────
    mask_arr = nib.load(tmp_mask).get_fdata().astype(bool)   # (H, W, D)

    # ── Step 3 & 4: apply mask to every modality, save to temp dir ───────────
    for key in image_keys:
        img      = nib.load(entry[key])
        stripped = img.get_fdata().astype(np.float32) * mask_arr
        out_path = tmp_dir / f'{key}_stripped.nii.gz'
        nib.save(nib.Nifti1Image(stripped, img.affine, img.header), str(out_path))
        new_entry[key] = str(out_path)

    new_entry['_tmpdir'] = str(tmp_dir)
    return new_entry


In [49]:
# ──  — Per-Model Transforms & Invertd ───────────────────────────────────

def _make_infer_tf(crop_key):
    """Transform pipeline for unlabelled cases (no seg key)."""
    return Compose([
        LoadImaged(keys=IMAGE_KEYS),
        EnsureChannelFirstd(keys=IMAGE_KEYS),
        Orientationd(keys=IMAGE_KEYS, axcodes='RAS'),
        NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
        CropForegroundd(keys=IMAGE_KEYS, source_key=crop_key, allow_smaller=True),
    ])

def _make_test_tf(crop_key):
    """Transform pipeline for labelled cases (includes seg key for GT load)."""
    return Compose([
        LoadImaged(keys=IMAGE_KEYS + ['seg']),
        EnsureChannelFirstd(keys=IMAGE_KEYS + ['seg']),
        Orientationd(keys=IMAGE_KEYS + ['seg'], axcodes='RAS'),
        NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
        CropForegroundd(keys=IMAGE_KEYS + ['seg'], source_key=crop_key, allow_smaller=True),
    ])

def _make_invertd(tf, crop_key):
    """Invertd that undoes tf for the 'pred' key using crop_key as affine reference."""
    return Invertd(
        keys='pred',
        transform=tf,
        orig_keys=crop_key,
        meta_keys='pred_meta_dict',
        orig_meta_keys=f'{crop_key}_meta_dict',
        meta_key_postfix='meta_dict',
        nearest_interp=True,
        to_tensor=True,
    )

# Build per-model transform pairs
tf_ed_infer  = _make_infer_tf('t2f');  tf_ed_test  = _make_test_tf('t2f')
tf_cc_infer  = _make_infer_tf('t2w');  tf_cc_test  = _make_test_tf('t2w')
tf_et_infer  = _make_infer_tf('t1c');  tf_et_test  = _make_test_tf('t1c')
tf_net_infer = _make_infer_tf('t1c');  tf_net_test = _make_test_tf('t1c')

invertd_ed_infer  = _make_invertd(tf_ed_infer,  't2f')
invertd_ed_test   = _make_invertd(tf_ed_test,   't2f')
invertd_cc_infer  = _make_invertd(tf_cc_infer,  't2w')
invertd_cc_test   = _make_invertd(tf_cc_test,   't2w')
invertd_et_infer  = _make_invertd(tf_et_infer,  't1c')
invertd_et_test   = _make_invertd(tf_et_test,   't1c')
invertd_net_infer = _make_invertd(tf_net_infer, 't1c')
invertd_net_test  = _make_invertd(tf_net_test,  't1c')



In [50]:
# ── Cell 7 — Inference Engine ─────────────────────────────────────────────────

def _infer_one(tf, model, entry, roi, sw_batch, overlap):
    
    ds   = Dataset(data=[entry], transform=tf)
    item = ds[0]
    inp  = torch.cat([item[k].unsqueeze(0).to(DEVICE) for k in IMAGE_KEYS], dim=1)
    with torch.no_grad(), autocast(enabled=USE_AMP):
        logits = sliding_window_inference(
            inp, roi, sw_batch, model, overlap=overlap,
            mode='gaussian', progress=False,
        )
    probs = F.softmax(logits, dim=1).squeeze(0).cpu()   # (C, D, H, W)
    return probs, item


def _inject_and_invert(item, pred_argmax, invertd_fn, crop_key):
   
    p = pred_argmax
    if p.dim() == 3:
        p = p.unsqueeze(0)   # (1, D, H, W)

    ref_mt = item[crop_key]
    pred_meta = MetaTensor(
        p.float().cpu(),
        meta=ref_mt.meta.copy() if hasattr(ref_mt, 'meta') else {},
        applied_operations=list(ref_mt.applied_operations)
            if hasattr(ref_mt, 'applied_operations') else [],
    )
    item['pred'] = pred_meta

    inv_batch = invertd_fn(item)

    pred_inv = inv_batch['pred']
    pred_np  = pred_inv.cpu().numpy() if isinstance(pred_inv, torch.Tensor) else np.asarray(pred_inv)
    pred_orig = np.squeeze(pred_np).astype(np.uint8)

    # Recover affine
    if hasattr(pred_inv, 'affine'):
        affine = pred_inv.affine.cpu().numpy()
    else:
        meta_dict = inv_batch.get('pred_meta_dict', {})
        ref_path  = meta_dict.get('filename_or_obj')
        affine    = nib.load(ref_path).affine if ref_path else np.eye(4)

    # Recover header
    meta_dict = inv_batch.get('pred_meta_dict', inv_batch.get(f'{crop_key}_meta_dict', {}))
    ref_path  = meta_dict.get('filename_or_obj')
    orig_hdr  = nib.load(ref_path).header.copy() if ref_path else nib.Nifti1Header()
    orig_hdr.set_data_dtype(np.uint8)

    return pred_orig, affine, orig_hdr


@torch.no_grad()
def ensemble_predict(entry, use_test_pipeline):
    
    tf_ed  = tf_ed_test   if use_test_pipeline else tf_ed_infer
    tf_cc  = tf_cc_test   if use_test_pipeline else tf_cc_infer
    tf_et  = tf_et_test   if use_test_pipeline else tf_et_infer
    tf_net = tf_net_test  if use_test_pipeline else tf_net_infer

    # ── Step 1: sliding-window inference per specialist ───────────────────────
    probs_ed,  item_ed  = _infer_one(tf_ed,  model_ed,  entry, ED_SW_ROI,  ED_SW_BATCH,  ED_SW_OVERLAP)
    probs_cc,  item_cc  = _infer_one(tf_cc,  model_cc,  entry, CC_SW_ROI,  CC_SW_BATCH,  CC_SW_OVERLAP)
    probs_et,  item_et  = _infer_one(tf_et,  model_et,  entry, ET_SW_ROI,  ET_SW_BATCH,  ET_SW_OVERLAP)
    probs_net, item_net = _infer_one(tf_net, model_net, entry, NET_SW_ROI, NET_SW_BATCH, NET_SW_OVERLAP)

    # ── Step 2: binary argmax in each model's own crop space ──────────────────
    # All models are binary (ch0=BG, ch1=region) → argmax gives 0/1 mask
    pred_ed_crop  = probs_ed.argmax(dim=0).long()   # 0/1
    pred_cc_crop  = probs_cc.argmax(dim=0).long()   # 0/1
    pred_et_crop  = probs_et.argmax(dim=0).long()   # 0/1
    pred_net_crop = probs_net.argmax(dim=0).long()  # 0/1

    # ── Step 3: invert each binary mask to original image space ───────────────
    # Each uses its own crop-source key so Invertd undoes the correct crop box
    ed_orig,  affine, orig_hdr = _inject_and_invert(copy.deepcopy(item_ed),  pred_ed_crop,  invertd_ed_test  if use_test_pipeline else invertd_ed_infer,  't2f')
    cc_orig,  _,      _        = _inject_and_invert(copy.deepcopy(item_cc),  pred_cc_crop,  invertd_cc_test  if use_test_pipeline else invertd_cc_infer,  't2w')
    et_orig,  _,      _        = _inject_and_invert(copy.deepcopy(item_et),  pred_et_crop,  invertd_et_test  if use_test_pipeline else invertd_et_infer,  't1c')
    net_orig, _,      _        = _inject_and_invert(copy.deepcopy(item_net), pred_net_crop, invertd_net_test if use_test_pipeline else invertd_net_infer, 't1c')

    # ── Step 4: combine in original space (priority: ET > NET > CC > ED) ──────
    final_label = np.zeros_like(ed_orig, dtype=np.uint8)
    final_label[ed_orig  == 1] = 4   # ED  — outermost
    final_label[cc_orig  == 1] = 3   # CC
    final_label[net_orig == 1] = 2   # NET
    final_label[et_orig  == 1] = 1   # ET  — innermost, highest priority

    return final_label, affine, orig_hdr

DEFAULT_MIN_VOXELS = {1: 100, 2: 200, 3: 500, 4: 130}

def remove_small_components(label_map, min_voxels_per_label=None):
    
    if min_voxels_per_label is None:
        min_voxels_per_label = DEFAULT_MIN_VOXELS
    cleaned = label_map.copy()
    struct  = ndimage.generate_binary_structure(3, 2)
    for lbl in [1, 2, 3, 4]:
        mask = (label_map == lbl)
        if not mask.any(): continue
        labeled_blobs, n_blobs = ndimage.label(mask, structure=struct)
        blob_sizes = ndimage.sum(mask, labeled_blobs, range(1, n_blobs + 1))
        threshold  = min_voxels_per_label.get(lbl, 10)
        for idx, size in enumerate(blob_sizes, start=1):
            if size < threshold:
                cleaned[labeled_blobs == idx] = 0
    return cleaned


def remove_distant_components(label_map, max_distance_mm=15.0, voxel_size_mm=1.0):
    
    cleaned = label_map.copy()
    struct  = ndimage.generate_binary_structure(3, 2)
    max_dist_vox = max_distance_mm / voxel_size_mm
    for lbl in [1, 2, 3, 4]:
        mask = (label_map == lbl)
        if not mask.any(): continue
        labeled_blobs, n_blobs = ndimage.label(mask, structure=struct)
        if n_blobs <= 1: continue
        blob_sizes  = ndimage.sum(mask, labeled_blobs, range(1, n_blobs + 1))
        largest_idx = int(np.argmax(blob_sizes)) + 1
        dist_from_main = ndimage.distance_transform_edt(~(labeled_blobs == largest_idx))
        for idx in range(1, n_blobs + 1):
            if idx == largest_idx: continue
            if dist_from_main[labeled_blobs == idx].min() > max_dist_vox:
                cleaned[labeled_blobs == idx] = 0
    return cleaned


def enforce_net_ed_dependency(label_map, ed_max_dist_from_net_mm=15.0, voxel_size_mm=1.0):
    
    cleaned = label_map.copy()
    net_mask = (cleaned == 2)   # NET = label 2
    ed_mask  = (cleaned == 4)   # ED  = label 4

    
    n_slices = cleaned.shape[2]
    for z in range(n_slices):
        if not net_mask[:, :, z].any():
            # No NET in this slice → ED here is anatomically implausible
            cleaned[:, :, z][cleaned[:, :, z] == 4] = 0

    # Recompute ED mask after rule 1 (some ED may have been suppressed)
    ed_mask_after_r1 = (cleaned == 4)

   
    if not net_mask.any():
        cleaned[ed_mask_after_r1] = 0
        return cleaned

    max_dist_vox = ed_max_dist_from_net_mm / voxel_size_mm

    # Distance transform: distance of every non-NET voxel to the nearest NET voxel
    dist_to_net = ndimage.distance_transform_edt(~net_mask)

    # Label surviving ED connected components and test each against the distance
    struct = ndimage.generate_binary_structure(3, 2)
    labeled_ed, n_blobs = ndimage.label(ed_mask_after_r1, structure=struct)

    for idx in range(1, n_blobs + 1):
        blob_voxels = labeled_ed == idx
        # Minimum distance from any voxel in this ED blob to any NET voxel
        min_dist = dist_to_net[blob_voxels].min()
        if min_dist > max_dist_vox:
            cleaned[blob_voxels] = 0

    return cleaned


def save_outputs(final_label, affine, orig_hdr, sid):
    def _save(arr, folder, name):
        nib.save(nib.Nifti1Image(arr.astype(np.uint8), affine, orig_hdr),
                 str(folder / name))
    _save(final_label,                          DIR_FULL, f'{sid}.nii.gz')
    #_save((final_label == 1).astype(np.uint8),  DIR_ET,   f'{sid}_et.nii.gz')
     #_save((final_label == 2).astype(np.uint8),  DIR_NET,  f'{sid}_net.nii.gz')
     #_save((final_label == 3).astype(np.uint8),  DIR_CC,   f'{sid}_cc.nii.gz')
    # _save((final_label == 4).astype(np.uint8),  DIR_ED,   f'{sid}_ed.nii.gz')


In [51]:
# ── TTA (Test-Time Augmentation) Post-Processing ──────────────────

# ── Global toggle ─────────────────────────────────────────────────────────────
TTA_ENABLED   = True        # set False to skip TTA and use plain ensemble
TTA_AUG_IDS   = [0,1,2,3,4,5,6,7,8,9]   # which augmentations to run
TTA_VOTE_FRAC = 0.5         # voxel fires if vote_count / N_augs > this fraction


# ── Augmentation helpers ───────────────────────────────────────────────────────

def _aug_volumes(volumes_dict, aug_id):
    
    if aug_id == 0:
        # Identity — no-op
        return {k: v.copy() for k, v in volumes_dict.items()}, lambda m: m.copy()

    elif aug_id in (1, 2, 3, 4):
        # Axial rotation around z-axis (axis=(0,1) in H×W×D indexing)
        angle = {1: 5.0, 2: -5.0, 3: 10.0, 4: -10.0}[aug_id]
        aug = {}
        for k, v in volumes_dict.items():
            aug[k] = scipy_rotate(v.astype(np.float32), angle,
                                  axes=(0, 1), reshape=False,
                                  order=1, mode='constant', cval=0.0)
        def _invert_rot(mask, _angle=-angle):
            return scipy_rotate(mask.astype(np.float32), _angle,
                                axes=(0, 1), reshape=False,
                                order=0, mode='constant', cval=0.0).round().astype(np.uint8)
        return aug, _invert_rot

    elif aug_id == 5:
        # Flip H axis (left-right in RAS)
        aug = {k: np.flip(v, axis=0).copy() for k, v in volumes_dict.items()}
        invert = lambda m: np.flip(m, axis=0).copy()
        return aug, invert

    elif aug_id == 6:
        # Flip W axis (anterior-posterior in RAS)
        aug = {k: np.flip(v, axis=1).copy() for k, v in volumes_dict.items()}
        invert = lambda m: np.flip(m, axis=1).copy()
        return aug, invert

    elif aug_id == 7:
        # Flip D axis (superior-inferior in RAS)
        aug = {k: np.flip(v, axis=2).copy() for k, v in volumes_dict.items()}
        invert = lambda m: np.flip(m, axis=2).copy()
        return aug, invert

    elif aug_id in (8, 9):
        # Intensity scale jitter — only affects image, not label maps
        scale = {8: 1.05, 9: 0.95}[aug_id]
        aug = {k: (v.astype(np.float32) * scale) for k, v in volumes_dict.items()}
        invert = lambda m: m.copy()   # spatial identity; only intensity changed
        return aug, invert

    else:
        raise ValueError(f'Unknown aug_id: {aug_id}')


def _save_aug_niftis(aug_vols, entry, tmp_dir, aug_id):
    
    tmp_dir.mkdir(parents=True, exist_ok=True)
    new_entry = {}
    for key in IMAGE_KEYS:
        orig_img = nib.load(entry[key])
        tmp_path = tmp_dir / f'aug{aug_id}_{key}.nii.gz'
        nib.save(
            nib.Nifti1Image(aug_vols[key].astype(np.float32),
                            orig_img.affine, orig_img.header),
            str(tmp_path)
        )
        new_entry[key] = str(tmp_path)
    if 'seg' in entry:
        new_entry['seg'] = entry['seg']   # GT unchanged — not augmented
    new_entry['subject_id'] = entry['subject_id']
    return new_entry


def _vote_masks(mask_list, vote_frac=0.5):
    
    n = len(mask_list)
    shape = mask_list[0].shape
    threshold = vote_frac * n   # float — count must be strictly greater

    # Accumulate per-label vote counts
    counts = {lbl: np.zeros(shape, dtype=np.int16) for lbl in [1, 2, 3, 4]}
    for mask in mask_list:
        for lbl in [1, 2, 3, 4]:
            counts[lbl] += (mask == lbl).astype(np.int16)

    # Build voted label map with priority merge (ET innermost = highest priority)
    voted = np.zeros(shape, dtype=np.uint8)
    for lbl in [4, 3, 2, 1]:   # ED first (overwritten by higher-priority later)
        voted[counts[lbl] > threshold] = lbl

    return voted


@torch.no_grad()
def tta_ensemble_predict(entry, use_test_pipeline, aug_ids=None,
                         vote_frac=None, tmp_subdir='_tta_tmp'):
    
    if aug_ids  is None: aug_ids   = TTA_AUG_IDS
    if vote_frac is None: vote_frac = TTA_VOTE_FRAC

    sid     = entry['subject_id']
    tmp_dir = OUTPUT_ROOT / tmp_subdir / sid

    # Load raw volumes once — we'll augment in-memory
    raw_vols = {}
    for key in IMAGE_KEYS:
        img = nib.load(entry[key])
        raw_vols[key] = img.get_fdata().astype(np.float32)   # (H, W, D)

    collected_masks = []   # list of (H,W,D) uint8 in original space
    affine   = None
    orig_hdr = None

    for aug_id in aug_ids:
        print(f'    TTA aug {aug_id:2d}/', end='', flush=True)

        # 1. Augment volumes
        aug_vols, invert_fn = _aug_volumes(raw_vols, aug_id)

        # 2. Write temp NIfTIs
        aug_entry = _save_aug_niftis(aug_vols, entry, tmp_dir, aug_id)

        # 3. Run ensemble
        pred_mask, _aff, _hdr = ensemble_predict(aug_entry,
                                                  use_test_pipeline=use_test_pipeline)
        if affine   is None: affine   = _aff
        if orig_hdr is None: orig_hdr = _hdr

        # 4. Invert augmentation on the predicted mask
        inv_mask = invert_fn(pred_mask)   # (H, W, D) uint8

        collected_masks.append(inv_mask)
        n_fg = int((inv_mask > 0).sum())
        print(f' done  (fg={n_fg:,})', flush=True)

        # Free temp VRAM
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # 5. Majority vote
  
    voted_label = _vote_masks(collected_masks, vote_frac=vote_frac)
  

    # 6. Post-processing on the voted map
    voted_label = remove_small_components(voted_label)
    voted_label = remove_distant_components(voted_label, max_distance_mm=15.0)
    voted_label = enforce_net_ed_dependency(voted_label, ed_max_dist_from_net_mm=15.0)

    # 7. Clean up temp NIfTI files
    import shutil
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir, ignore_errors=True)

    return voted_label, affine, orig_hdr

_AUG_DESC = {
    0: 'Identity (base prediction)',
    1: 'Axial rotation  +5°',
    2: 'Axial rotation  −5°',
    3: 'Axial rotation +10°',
    4: 'Axial rotation −10°',
    5: 'Flip H-axis (L/R)',
    6: 'Flip W-axis (A/P)',
    7: 'Flip D-axis (S/I)',
    8: 'Intensity scale ×1.05',
    9: 'Intensity scale ×0.95',
}
for k, v in _AUG_DESC.items():
    flag = '✓' if k in TTA_AUG_IDS else '✗'
   


In [52]:
if not unlabeled_dicts:
    pass
else:
    t_total = time.time()
    results = []

    for idx, entry in enumerate(unlabeled_dicts, 1):
        sid = entry['subject_id']
        t0 = time.time()

        stripped_entry = deskull_case(entry)

        try:
            if TTA_ENABLED:
                final_label, affine, orig_hdr = tta_ensemble_predict(
                    stripped_entry, use_test_pipeline=('seg' in entry)
                )
            else:
                final_label, affine, orig_hdr = ensemble_predict(
                    stripped_entry, use_test_pipeline=('seg' in entry)
                )
                final_label = remove_small_components(final_label)
                final_label = remove_distant_components(final_label, max_distance_mm=15.0)
                final_label = enforce_net_ed_dependency(
                    final_label, ed_max_dist_from_net_mm=15.0
                )
        finally:
            shutil.rmtree(stripped_entry.get('_tmpdir', ''), ignore_errors=True)

        save_outputs(final_label, affine, orig_hdr, sid)


        elapsed = round(time.time() - t0, 1)
        vox = {l: int((final_label == l).sum()) for l in range(NUM_CLASSES)}

        results.append({
            'subject_id': sid,
             **{f'vox_{LABEL_NAMES[l]}': vox[l] for l in range(NUM_CLASSES)},
             'time_s': elapsed
})

        del final_label
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()



    TTA aug  0/ done  (fg=75,072)
    TTA aug  1/ done  (fg=79,277)
    TTA aug  2/ done  (fg=80,978)
    TTA aug  3/ done  (fg=79,151)
    TTA aug  4/ done  (fg=82,774)
    TTA aug  5/ done  (fg=75,769)
    TTA aug  6/ done  (fg=74,885)
    TTA aug  7/ done  (fg=84,858)
    TTA aug  8/ done  (fg=75,077)
    TTA aug  9/ done  (fg=75,075)
    TTA aug  0/ done  (fg=45,722)
    TTA aug  1/ done  (fg=45,792)
    TTA aug  2/ done  (fg=45,657)
    TTA aug  3/ done  (fg=45,918)
    TTA aug  4/ done  (fg=45,560)
    TTA aug  5/ done  (fg=47,503)
    TTA aug  6/ done  (fg=45,262)
    TTA aug  7/ done  (fg=44,972)
    TTA aug  8/ done  (fg=45,720)
    TTA aug  9/ done  (fg=45,722)
    TTA aug  0/ done  (fg=34,774)
    TTA aug  1/ done  (fg=27,811)
    TTA aug  2/ done  (fg=31,669)
    TTA aug  3/ done  (fg=30,661)
    TTA aug  4/ done  (fg=29,022)
    TTA aug  5/ done  (fg=35,870)
    TTA aug  6/ done  (fg=33,855)
    TTA aug  7/ done  (fg=36,960)
    TTA aug  8/ done  (fg=34,773)
    TTA aug  9